# Exploratory Data Analysis — ElderGuard Analytics
## Gas Monitoring & Activity Level Classification

**Objective:** Understand the structure, quality, and patterns in the smart home sensor dataset to support building predictive models for classifying elderly resident activity levels.

**Dataset:** `gas_monitoring.db` — 10,000 rows of environmental sensor readings from elderly residents' homes.

**Target Variable:** `Activity Level` — classifies resident activity as Low, Moderate, or High.

---

### EDA Process Overview
1. Data Loading & Initial Inspection
2. Data Quality Assessment (Nulls, Duplicates, Dirty Labels)
3. Data Cleaning & Standardisation
4. Univariate Analysis
5. Bivariate Analysis (Features vs Target)
6. Correlation & Multicollinearity Analysis
7. Session-Level Analysis
8. Summary & Implications for Modelling

---
## Step 1: Data Loading & Initial Inspection

**Purpose:** Load the dataset from the SQLite database and get a high-level understanding of its shape, column types, and sample values before any analysis.

We use SQLite3 to connect and Pandas to load the data into a DataFrame.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)
PALETTE = {'Low Activity': '#4C72B0', 'Moderate Activity': '#DD8452', 'High Activity': '#55A868'}

# Load data
conn = sqlite3.connect('data/gas_monitoring.db')
df_raw = pd.read_sql('SELECT * FROM gas_monitoring', conn)
conn.close()

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(5)

In [ ]:
df_raw.dtypes

**Conclusion — Step 1:**
- The dataset has **10,000 rows** and **14 columns**: 9 continuous sensor readings, 4 categorical columns, and 1 integer session ID.
- Numeric features are stored as `REAL` (float), categorical features as `TEXT`.
- No obvious structural issues at this stage — deeper quality checks follow.

---
## Step 2: Data Quality Assessment

**Purpose:** Identify missing values, duplicate rows, and label inconsistencies. The problem statement explicitly warns the dataset may contain *synthetic or contaminated data*, so this step is critical before any analysis.

We check:
- Missing values per column
- Duplicate rows
- Label noise in categorical columns (e.g. mixed casing, typos)

In [ ]:
# --- Missing values ---
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('=== Missing Values ===')
print(missing_df.to_string())

print(f'\n=== Duplicate Rows ===')
print(f'  {df_raw.duplicated().sum()} duplicate rows found')

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(9, 4))
missing_df['Missing %'].sort_values().plot(kind='barh', ax=ax, color='#c0392b')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Rate per Column')
for p in ax.patches:
    ax.annotate(f'{p.get_width():.1f}%', (p.get_width() + 0.3, p.get_y() + 0.3))
plt.tight_layout()
plt.show()

In [ ]:
# --- Label noise: Activity Level ---
print('Raw Activity Level values:')
print(df_raw['Activity Level'].value_counts().to_string())

print('\nRaw HVAC Operation Mode values:')
print(df_raw['HVAC Operation Mode'].value_counts().to_string())

**Conclusion — Step 2:**
- **4 columns have missing values:** `Humidity` (~19%), `MetalOxideSensor_Unit2` (~14%), `Ambient Light Level` (~11%), `CO_GasSensor` (~8%).
- **No duplicate rows** detected.
- **Significant label noise** found in `Activity Level` — values like `Low_Activity`, `LowActivity`, `ModerateActivity` are inconsistent spellings of the same class. These must be standardised.
- **HVAC Operation Mode** also has mixed casing (e.g. `COOLING_ACTIVE`, `Cooling_Active`, `cooling_active`) — all referring to the same category.

**Assumption:** `Low_Activity`, `LowActivity` → `Low Activity`; `ModerateActivity` → `Moderate Activity`. All HVAC labels normalised to lowercase with underscores.

---
## Step 3: Data Cleaning & Standardisation

**Purpose:** Apply all corrections identified in Step 2, and handle temperature outliers identified through domain knowledge (indoor temperature above ~40°C or below 10°C is physically implausible for a living space).

We will:
- Standardise all categorical labels
- Cap/flag implausible temperature values
- Impute missing numeric values with the column median (robust to outliers)
- Fill missing `Ambient Light Level` with mode

In [ ]:
df = df_raw.copy()

# --- 1. Standardise Activity Level labels ---
activity_map = {
    'Low Activity': 'Low Activity',
    'Low_Activity': 'Low Activity',
    'LowActivity': 'Low Activity',
    'Moderate Activity': 'Moderate Activity',
    'ModerateActivity': 'Moderate Activity',
    'High Activity': 'High Activity',
}
df['Activity Level'] = df['Activity Level'].map(activity_map)

# --- 2. Standardise HVAC labels: lowercase + underscore ---
df['HVAC Operation Mode'] = (
    df['HVAC Operation Mode']
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
)

# --- 3. Flag temperature outliers (outside 10-40 C for indoor living) ---
TEMP_LOW, TEMP_HIGH = 10.0, 40.0
temp_outlier_mask = (df['Temperature'] < TEMP_LOW) | (df['Temperature'] > TEMP_HIGH)
print(f'Temperature outliers: {temp_outlier_mask.sum()} rows ({temp_outlier_mask.mean()*100:.1f}%)')
print(f'  Min: {df["Temperature"].min():.2f}, Max: {df["Temperature"].max():.2f}')

# Impute temperature outliers with median of non-outlier values
temp_median = df.loc[~temp_outlier_mask, 'Temperature'].median()
df.loc[temp_outlier_mask, 'Temperature'] = temp_median
print(f'  Replaced with median: {temp_median:.2f} C')

# --- 4. Impute missing numeric columns with median ---
numeric_cols_with_nulls = ['Humidity', 'MetalOxideSensor_Unit2', 'CO_GasSensor']
for col in numeric_cols_with_nulls:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f'  Imputed {col} nulls with median={median_val:.2f}')

# --- 5. Fill Ambient Light Level with mode ---
light_mode = df['Ambient Light Level'].mode()[0]
df['Ambient Light Level'].fillna(light_mode, inplace=True)
print(f'  Imputed Ambient Light Level nulls with mode="{light_mode}"')

print(f'\nRemaining nulls: {df.isnull().sum().sum()}')
print('\nCleaned Activity Level distribution:')
print(df['Activity Level'].value_counts())

**Conclusion — Step 3:**
- Activity Level and HVAC labels are now clean and consistent.
- Temperature outliers (e.g. 307°C, 292°C) are clearly erroneous sensor readings/synthetic contamination — replaced with median indoor temperature.
- Missing numeric values imputed with column median; `Ambient Light Level` filled with mode.
- Dataset is now clean with **zero remaining null values**.

---
## Step 4: Univariate Analysis

**Purpose:** Understand the individual distribution of each feature in isolation — its range, central tendency, spread, and skewness. This helps identify further anomalies and informs feature engineering decisions.

We examine:
- Summary statistics for all numeric features
- Distribution plots for each numeric sensor
- Count plots for all categorical features

In [ ]:
numeric_cols = [
    'Temperature', 'Humidity', 'CO2_InfraredSensor', 'CO2_ElectroChemicalSensor',
    'MetalOxideSensor_Unit1', 'MetalOxideSensor_Unit2', 'MetalOxideSensor_Unit3',
    'MetalOxideSensor_Unit4', 'CO_GasSensor'
]

desc = df[numeric_cols].describe().T
desc['skewness'] = df[numeric_cols].skew().round(3)
desc['kurtosis'] = df[numeric_cols].kurt().round(3)
desc.round(3)

In [ ]:
# Distribution plots for all numeric sensors
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.histplot(df[col], bins=50, kde=True, ax=ax, color='#4C72B0', alpha=0.7)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    skew = df[col].skew()
    ax.annotate(f'skew={skew:.2f}', xy=(0.97, 0.92), xycoords='axes fraction',
                ha='right', fontsize=9, color='#c0392b')

plt.suptitle('Distribution of Numeric Sensor Features (post-cleaning)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature distributions
cat_cols = ['Activity Level', 'Time of Day', 'HVAC Operation Mode', 'Ambient Light Level']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, col in enumerate(cat_cols):
    ax = axes[i]
    vc = df[col].value_counts()
    vc.plot(kind='bar', ax=ax, color=sns.color_palette('muted', len(vc)), edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height() + 10),
                    ha='center', fontsize=8)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Conclusion — Step 4:**
- **CO2 sensors** show near-normal distributions centred around expected indoor CO2 levels (~550–580 ppm), indicating plausible readings.
- **Metal Oxide Sensors** (Units 1–4) appear approximately normal, typical for gas sensors in a stable indoor environment.
- **CO_GasSensor** is right-skewed — most readings are low (safe), with occasional spikes that could signal cooking or poor ventilation events.
- **Temperature** is now tightly distributed around ~20°C post-cleaning, confirming the outliers were indeed erroneous.
- **Activity Level** is **imbalanced** — Low Activity dominates (~57%), followed by Moderate (~31%), then High (~12%). This imbalance must be addressed during modelling (e.g. class weights or oversampling).
- All four time-of-day periods are roughly equally represented (~24–26% each), ensuring no temporal bias.

---
## Step 5: Bivariate Analysis — Features vs Activity Level

**Purpose:** Understand how each feature relates to the target variable `Activity Level`. Features that show strong separation between activity classes are likely to be informative predictors in the ML models.

We use:
- Box plots for numeric features grouped by activity level
- Stacked bar charts for categorical features vs activity level

In [ ]:
activity_order = ['Low Activity', 'Moderate Activity', 'High Activity']

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.boxplot(
        data=df, x='Activity Level', y=col, order=activity_order,
        palette=PALETTE, ax=ax, width=0.5, flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Sensor Readings by Activity Level', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Activity Level
cat_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level']
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, col in enumerate(cat_features):
    ax = axes[i]
    ct = pd.crosstab(df[col], df['Activity Level'], normalize='index') * 100
    ct = ct[[c for c in activity_order if c in ct.columns]]
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=[PALETTE[c] for c in ct.columns], edgecolor='white', width=0.7)
    ax.set_title(f'{col} vs Activity Level', fontweight='bold')
    ax.set_ylabel('% of rows')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Categorical Features vs Activity Level (normalised)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for CO2 sensors - often strongest signal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
co2_cols = ['CO2_InfraredSensor', 'CO2_ElectroChemicalSensor']

for i, col in enumerate(co2_cols):
    sns.violinplot(
        data=df, x='Activity Level', y=col, order=activity_order,
        palette=PALETTE, ax=axes[i], inner='quartile'
    )
    axes[i].set_title(f'{col} by Activity Level', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('CO2 Sensor Distributions by Activity Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Group means table
group_means = df.groupby('Activity Level')[numeric_cols].mean().round(2)
group_means.loc[activity_order]

**Conclusion — Step 5:**
- **CO2 sensors** (both infrared and electrochemical) show clear upward trends with activity level — higher activity means more CO2 production from respiration and movement. This makes them strong candidate features.
- **CO_GasSensor** shows higher readings during High Activity, possibly from cooking or appliance use.
- **Metal Oxide Sensors** show moderate separation — Units 1 and 3 appear more discriminative than Units 2 and 4.
- **Temperature and Humidity** show limited separation across classes — they may still contribute when combined with other features.
- **Ambient Light Level** shows a notable pattern: High Activity correlates with brighter environments (awake/active) while Low Activity correlates with dim/very dim lighting (resting/sleeping).
- **Time of Day** shows that High Activity is proportionally more frequent in the morning and afternoon periods, which is intuitive.

---
## Step 6: Correlation & Multicollinearity Analysis

**Purpose:** Identify which features are strongly correlated with each other (multicollinearity) and which are most correlated with the target. Highly correlated feature pairs carry redundant information — important context for feature selection.

We use a Pearson correlation heatmap on numeric features, and encode the target ordinally for correlation analysis.

In [ ]:
# Encode target ordinally for correlation computation
activity_encode = {'Low Activity': 0, 'Moderate Activity': 1, 'High Activity': 2}
df['Activity_Encoded'] = df['Activity Level'].map(activity_encode)

corr_cols = numeric_cols + ['Activity_Encoded']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Pearson Correlation Matrix (Numeric Features + Encoded Target)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with target - ranked
target_corr = corr_matrix['Activity_Encoded'].drop('Activity_Encoded').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
target_corr.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_r', len(target_corr)))
ax.set_title('Feature Correlation with Activity Level (absolute Pearson r)', fontweight='bold')
ax.set_ylabel('|Pearson r|')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=35)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width()/2, p.get_height() + 0.002),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Feature correlations with target (ranked):')
print(target_corr.to_string())

**Conclusion — Step 6:**
- **CO2_ElectroChemicalSensor** has the highest correlation with the target (~0.25–0.35), confirming it as a top predictor.
- **CO2_InfraredSensor** and **CO2_ElectroChemicalSensor** are highly correlated with each other (expected — they measure the same gas with different technologies). One could be dropped without significant information loss, though tree-based models handle redundancy naturally.
- **Metal Oxide Sensor Units 1–4** show moderate inter-correlation, suggesting they pick up similar volatile compound signals.
- **Temperature and Humidity** have low target correlation — they may still contribute non-linearly but are less important individually.
- No near-perfect collinearity (r > 0.95) — all features can be retained for modelling.

---
## Step 7: Session-Level Analysis

**Purpose:** The dataset contains a `Session ID` identifying each monitoring session. Understanding whether sessions differ in their characteristics helps detect batch effects or data quality issues at the session level, and informs whether session should be used as a feature or stratification variable.

In [ ]:
session_stats = df.groupby('Session ID').agg(
    rows=('Activity Level', 'count'),
    low_pct=('Activity_Encoded', lambda x: (x == 0).mean() * 100),
    mod_pct=('Activity_Encoded', lambda x: (x == 1).mean() * 100),
    high_pct=('Activity_Encoded', lambda x: (x == 2).mean() * 100),
    avg_co2=('CO2_ElectroChemicalSensor', 'mean'),
    avg_temp=('Temperature', 'mean'),
).reset_index()

print(f'Unique sessions: {df["Session ID"].nunique()}')
print(f'Rows per session - min: {session_stats["rows"].min()}, max: {session_stats["rows"].max()}, mean: {session_stats["rows"].mean():.1f}')
session_stats.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rows per session distribution
session_stats['rows'].plot(kind='hist', bins=30, ax=axes[0], color='#4C72B0', edgecolor='white')
axes[0].set_title('Distribution of Rows per Session', fontweight='bold')
axes[0].set_xlabel('Number of rows')

# Average CO2 per session
axes[1].scatter(session_stats['Session ID'], session_stats['avg_co2'],
                alpha=0.5, s=15, color='#DD8452')
axes[1].set_title('Mean CO2 (ElectroChemical) per Session', fontweight='bold')
axes[1].set_xlabel('Session ID')
axes[1].set_ylabel('Mean CO2 (ppm)')

plt.tight_layout()
plt.show()

In [ ]:
# Activity distribution across sessions (sample of 20 sessions)
sample_sessions = df['Session ID'].value_counts().head(20).index
activity_by_session = (
    df[df['Session ID'].isin(sample_sessions)]
    .groupby(['Session ID', 'Activity Level'])
    .size()
    .unstack(fill_value=0)
)
activity_by_session_pct = activity_by_session.div(activity_by_session.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 5))
activity_by_session_pct[[c for c in activity_order if c in activity_by_session_pct.columns]].plot(
    kind='bar', stacked=True, ax=ax,
    color=[PALETTE[c] for c in activity_order if c in activity_by_session_pct.columns],
    edgecolor='white'
)
ax.set_title('Activity Level Distribution across Sessions (top 20)', fontweight='bold')
ax.set_xlabel('Session ID')
ax.set_ylabel('% of readings')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Conclusion — Step 7:**
- The dataset spans multiple sessions of varying lengths — some sessions have only a handful of readings, others have dozens.
- Most sessions are dominated by Low Activity readings, consistent with the overall class distribution.
- CO2 levels vary across sessions, likely reflecting different residents or different time periods of monitoring.
- Session ID should **not** be used as a predictive feature (it is an identifier, not a measurement). However, it is useful for understanding data grouping and could be used for grouped cross-validation during modelling.

---
## Step 8: Summary & Implications for Modelling

**Purpose:** Synthesise all EDA findings into actionable insights that will directly guide the modelling pipeline.

In [ ]:
# Final cleaned dataset summary
print('=== Final Cleaned Dataset ===')
print(f'Shape: {df.shape}')
print(f'Missing values: {df.drop(columns=["Activity_Encoded"]).isnull().sum().sum()}')
print()
print('Activity Level distribution (final):')
vc = df['Activity Level'].value_counts()
for k, v in vc.items():
    print(f'  {k}: {v} ({v/len(df)*100:.1f}%)')

In [ ]:
# Save cleaned dataset for downstream modelling
df.drop(columns=['Activity_Encoded']).to_csv('data/cleaned_gas_monitoring.csv', index=False)
print('Cleaned dataset saved to data/cleaned_gas_monitoring.csv')

---
## EDA Summary

### Key Findings

| Finding | Detail |
|---|---|
| **Label noise** | `Activity Level` and `HVAC Operation Mode` had inconsistent casing/formatting — standardised |
| **Missing values** | 4 columns had 8–19% missing — imputed with median/mode |
| **Temperature outliers** | Readings up to 307°C — physically impossible indoors, replaced with median (~20°C) |
| **Class imbalance** | Low:Moderate:High = 57%:31%:12% — models must use class weights or SMOTE |
| **Top predictors** | CO2 sensors (both), CO_GasSensor, Metal Oxide Units 1 & 3, Ambient Light Level |
| **Redundant features** | CO2 infrared and electrochemical are highly correlated — consider retaining both for tree models |
| **Session structure** | Data is grouped by session — use stratified/group cross-validation |

### Recommended Models
Given the class imbalance, mixed feature types (numeric + categorical), and non-linear sensor relationships:
1. **Random Forest** — handles mixed types, robust to outliers, provides feature importance
2. **Gradient Boosted Trees (XGBoost/LightGBM)** — best performance on tabular imbalanced data
3. **Logistic Regression (multinomial)** — interpretable baseline with class weights

### Pre-modelling Steps
- Encode categorical features: `Time of Day`, `HVAC Operation Mode`, `Ambient Light Level` → ordinal or one-hot
- Apply class weights (`class_weight='balanced'`) or SMOTE oversampling for minority class
- Use stratified K-fold cross-validation
- Evaluate with macro F1-score (appropriate for imbalanced multiclass)